In [19]:
pip install mplcursors

Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"
OUT_FILE = BASE_DIR / "tvpvar_input_scaled_dcmetals.csv"

KEEP_COLS = [
    "Date",
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
]

df = pd.read_csv(INPUT_FILE)

missing = [c for c in KEEP_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

out = df[KEEP_COLS].copy()
out.to_csv(OUT_FILE, index=False)

print(f"Saved: {OUT_FILE}")
print(out.head())

Saved: tvpvar_input_scaled_dcmetals.csv
         Date  dlog_SOLVPN  dlog_COPPER  dlog_GOLD  dlog_SILVER
0  2020-10-13    -0.408673    -0.388006  -1.687144    -2.111589
1  2020-10-14    -1.083456     0.095267   0.536998     0.453141
2  2020-10-15    -0.060821     0.641368   0.009273    -0.353759
3  2020-10-19    -0.419849     0.329228   0.174788     0.830391
4  2020-10-20     0.082134     1.132160   0.108119     0.470602


In [21]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# TVP-VAR + GFEVD for reduced variables
# variables:
#   dlog_SOLVPN, dlog_COPPER, dlog_GOLD, dlog_SILVER
# ============================================================

BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled_dcmetals.csv"

OUT_BETA = BASE_DIR / "tvpvar_beta_dcmetals.npy"
OUT_COV = BASE_DIR / "tvpvar_cov_dcmetals.npy"
OUT_GFEVD = BASE_DIR / "gfevd_all_dcmetals.npy"
OUT_LAG = BASE_DIR / "tvpvar_selected_lag_dcmetals.txt"

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
]

# horizon
H = 10

# Kalman params
LAMBDA = 0.99
DELTA = 0.01


def select_lag(df_num, maxlags=5):
    model = VAR(df_num)
    sel = model.select_order(maxlags=maxlags)
    # 기존 선호대로 BIC
    p = sel.selected_orders["bic"]
    if p is None or p < 1:
        p = 1
    return p


def build_lagged_xy(df_num, p):
    Y = df_num.values[p:]
    X_list = []
    for t in range(p, len(df_num)):
        row = []
        for lag in range(1, p + 1):
            row.extend(df_num.values[t - lag])
        X_list.append(row)
    X = np.asarray(X_list)
    return Y, X


def tvp_var_kalman(Y, X, k, p, lam=0.99, delta=0.01):
    """
    equation-by-equation TVP-VAR
    Y: (T, k)
    X: (T, k*p)
    returns:
      beta_t: (T, k, k*p)
      cov_t : (T, k, k)
    """
    T = Y.shape[0]
    state_dim = X.shape[1]

    beta_t = np.zeros((T, k, state_dim))
    resid_t = np.zeros((T, k))

    for eq in range(k):
        beta_prev = np.zeros(state_dim)
        P_prev = np.eye(state_dim) * 0.1
        R = delta

        for t in range(T):
            x_t = X[t]
            y_t = Y[t, eq]

            # predict
            beta_pred = beta_prev.copy()
            P_pred = P_prev / lam

            # update
            y_hat = x_t @ beta_pred
            e_t = y_t - y_hat

            S_t = x_t @ P_pred @ x_t.T + R
            K_t = (P_pred @ x_t) / S_t

            beta_upd = beta_pred + K_t * e_t
            P_upd = P_pred - np.outer(K_t, x_t) @ P_pred

            beta_t[t, eq, :] = beta_upd
            resid_t[t, eq] = e_t

            beta_prev = beta_upd
            P_prev = P_upd

    # rolling residual covariance
    cov_t = np.zeros((T, k, k))
    for t in range(T):
        start = max(0, t - 19)
        E = resid_t[start:t+1]
        if len(E) < 2:
            cov_t[t] = np.eye(k) * 1e-6
        else:
            c = np.cov(E.T)
            if c.ndim == 0:
                c = np.eye(k) * float(c)
            cov_t[t] = c + np.eye(k) * 1e-8

    return beta_t, cov_t


def beta_to_var_mats(beta_row, k, p):
    """
    beta_row: (k, k*p)
    return list of A_l matrices, each (k, k)
    """
    mats = []
    for lag in range(p):
        A = beta_row[:, lag*k:(lag+1)*k]
        mats.append(A)
    return mats


def tvp_vma_matrices(A_list, H):
    """
    A_list: list of p matrices A1..Ap, each (k,k)
    return Phi_0..Phi_{H-1}
    """
    p = len(A_list)
    k = A_list[0].shape[0]
    Phi = [np.eye(k)]

    for h in range(1, H):
        ph = np.zeros((k, k))
        for j in range(1, min(h, p) + 1):
            ph += Phi[h - j] @ A_list[j - 1]
        Phi.append(ph)
    return Phi


def generalized_fevd(A_list, Sigma, H):
    """
    Koop-Pesaran-Shin generalized FEVD
    return (k,k), row-normalized later
    """
    k = Sigma.shape[0]
    Phi = tvp_vma_matrices(A_list, H)

    fevd = np.zeros((k, k))
    sigma_diag = np.diag(Sigma).copy()
    sigma_diag = np.where(np.abs(sigma_diag) < 1e-12, 1e-12, sigma_diag)

    for i in range(k):
        denom = 0.0
        for h in range(H):
            denom += (Phi[h] @ Sigma @ Phi[h].T)[i, i]

        denom = max(denom, 1e-12)

        for j in range(k):
            numer = 0.0
            e_j = np.zeros(k)
            e_j[j] = 1.0

            for h in range(H):
                term = Phi[h] @ Sigma @ e_j
                numer += (term[i] ** 2)

            fevd[i, j] = numer / sigma_diag[j] / denom

    # row normalize
    row_sum = fevd.sum(axis=1, keepdims=True)
    row_sum = np.where(np.abs(row_sum) < 1e-12, 1e-12, row_sum)
    fevd = fevd / row_sum

    return fevd


# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df_num = df[VAR_NAMES].copy()

# -----------------------------
# Lag
# -----------------------------
p = select_lag(df_num, maxlags=5)
with open(OUT_LAG, "w", encoding="utf-8") as f:
    f.write(str(p))

print("Selected lag:", p)

# -----------------------------
# Build lagged data
# -----------------------------
Y, X = build_lagged_xy(df_num, p)
k = len(VAR_NAMES)

# -----------------------------
# TVP-VAR
# -----------------------------
beta_t, cov_t = tvp_var_kalman(Y, X, k=k, p=p, lam=LAMBDA, delta=DELTA)

np.save(OUT_BETA, beta_t)
np.save(OUT_COV, cov_t)

print("beta_t shape:", beta_t.shape)
print("cov_t shape :", cov_t.shape)

# -----------------------------
# GFEVD
# -----------------------------
T_eff = beta_t.shape[0]
gfevd_all = np.zeros((T_eff, k, k))

for t in range(T_eff):
    A_list = beta_to_var_mats(beta_t[t], k=k, p=p)
    Sigma = cov_t[t]
    gfevd_all[t] = generalized_fevd(A_list, Sigma, H=H)

np.save(OUT_GFEVD, gfevd_all)

print("Saved:")
print("-", OUT_BETA)
print("-", OUT_COV)
print("-", OUT_GFEVD)
print("-", OUT_LAG)

Selected lag: 1
beta_t shape: (1035, 4, 4)
cov_t shape : (1035, 4, 4)
Saved:
- tvpvar_beta_dcmetals.npy
- tvpvar_cov_dcmetals.npy
- gfevd_all_dcmetals.npy
- tvpvar_selected_lag_dcmetals.txt


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ----------------------------
# 1) Load
# ----------------------------
df = pd.read_csv("./tvpvar_input_scaled_dcmetals.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

gfevd = np.load("./gfevd_all_dcmetals.npy")

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
]

# ----------------------------
# 2) 날짜 길이 정렬
# ----------------------------
if len(df) > gfevd.shape[0]:
    diff = len(df) - gfevd.shape[0]
    df = df.iloc[diff:].reset_index(drop=True)
elif len(df) < gfevd.shape[0]:
    raise ValueError(
        f"Date rows ({len(df)}) < gfevd rows ({gfevd.shape[0]})."
    )

dates = df["Date"].copy()

# ----------------------------
# 3) 시각화 시작 시점
# ----------------------------
PLOT_START_DATE = "2021-01-01"
plot_mask = dates >= pd.to_datetime(PLOT_START_DATE)

# ----------------------------
# 4) TCI / NET 계산
# ----------------------------
def compute_tci(g):
    k = g.shape[1]
    diag = np.einsum("tii->t", g)
    offdiag = g.sum(axis=(1, 2)) - diag
    return offdiag / k * 100.0

def compute_net(g):
    T, k, _ = g.shape
    net = np.zeros((T, k))
    for t in range(T):
        m = g[t] * 100.0
        diag = np.diag(m)
        from_others = m.sum(axis=1) - diag
        to_others = m.sum(axis=0) - diag
        net[t] = to_others - from_others
    return net

tci = compute_tci(gfevd)
net = compute_net(gfevd)

plot_dates = dates[plot_mask].reset_index(drop=True)
plot_tci = tci[plot_mask.values]
plot_net = net[plot_mask.values]

# ----------------------------
# 5) Event 정의
# ----------------------------
events = [
    # AI / Data Center
    ("2022-11-30", "ChatGPT", "ai"),
    ("2023-01-23", "MSFT-OpenAI", "ai"),
    ("2023-03-14", "GPT-4", "ai"),
    ("2023-11-06", "OpenAI DevDay", "ai"),
    ("2024-01-01", "AI investment", "ai"),
    ("2024-03-18", "Blackwell", "ai"),
    ("2024-05-01", "Hyperscaler capex", "ai"),
    ("2024-07-31", "BigTech AI capex", "ai"),
    ("2024-10-15", "OCP Blackwell", "ai"),
    ("2025-01-21", "Stargate", "ai"),

    # Copper / Supply
    ("2021-11-15", "Copper shortage", "copper"),
    ("2023-03-01", "Copper bullish call", "copper"),
    ("2023-11-28", "Panama mine", "copper"),
    ("2024-08-15", "LME copper low", "copper"),
    ("2025-02-25", "US copper probe", "copper"),
    ("2025-03-10", "Copper policy risk", "copper"),

    # Infra / System
    ("2021-04-01", "Cloud demand", "infra"),
    ("2021-10-28", "Meta rename", "infra"),
    ("2022-03-22", "Hopper/H100", "infra"),
    ("2022-06-15", "Supply chain", "infra"),
    ("2022-09-15", "Energy spike", "infra"),
    ("2022-09-20", "H100 production", "infra"),
    ("2023-05-01", "AI infra demand", "infra"),
    ("2023-07-01", "GPU shortage", "infra"),
    ("2023-09-15", "DC deal peak", "infra"),
    ("2023-10-15", "Liquid cooling", "infra"),
    ("2024-09-20", "Power contract", "infra"),
]

COLOR_MAP = {
    "ai": "blue",
    "copper": "red",
    "infra": "green",
}

events = [(pd.to_datetime(d), label, cat) for d, label, cat in events]
events = [e for e in events if e[0] >= pd.to_datetime(PLOT_START_DATE)]
events = sorted(events, key=lambda x: x[0])

# ----------------------------
# 6) Event shape + annotation helper
# ----------------------------
def add_event_lines_and_labels(fig, events, y_min, y_max):
    yr = y_max - y_min
    levels = [y_max - yr * (0.04 + i * 0.09) for i in range(8)]

    for i, (dt, label, cat) in enumerate(events):
        color = COLOR_MAP[cat]

        fig.add_shape(
            type="line",
            x0=dt,
            x1=dt,
            y0=y_min,
            y1=y_max,
            line=dict(color=color, width=1.1, dash="dot")
        )

        fig.add_annotation(
            x=dt,
            y=levels[i % len(levels)],
            text=label,
            showarrow=False,
            textangle=-90,
            font=dict(size=10, color=color),
            xanchor="right",
            yanchor="top"
        )

def add_event_legend(fig):
    for cat, color in COLOR_MAP.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                line=dict(color=color, width=2, dash="dot"),
                name=f"event_{cat}",
                hoverinfo="skip",
                showlegend=True
            )
        )

# ----------------------------
# 7) TCI figure
# ----------------------------
fig_tci = go.Figure()

fig_tci.add_trace(
    go.Scatter(
        x=plot_dates,
        y=plot_tci,
        mode="lines",
        name="TCI"
    )
)

tci_ymin = float(np.min(plot_tci))
tci_ymax = float(np.max(plot_tci))

add_event_lines_and_labels(fig_tci, events, tci_ymin, tci_ymax)
add_event_legend(fig_tci)

fig_tci.update_layout(
    title="TCI (SOLVPN + Metals)",
    xaxis_title="Date",
    yaxis_title="TCI",
    height=700,
    hovermode="x unified"
)

fig_tci.show()

# ----------------------------
# 8) NET figure
# ----------------------------
fig_net = go.Figure()

for i, var in enumerate(VAR_NAMES):
    fig_net.add_trace(
        go.Scatter(
            x=plot_dates,
            y=plot_net[:, i],
            mode="lines",
            name=var
        )
    )

net_ymin = float(np.min(plot_net))
net_ymax = float(np.max(plot_net))

add_event_lines_and_labels(fig_net, events, net_ymin, net_ymax)
add_event_legend(fig_net)

fig_net.add_hline(y=0, line_width=1, line_color="black")

fig_net.update_layout(
    title="NET Spillover (SOLVPN + Metals)",
    xaxis_title="Date",
    yaxis_title="NET Spillover",
    height=750,
    hovermode="x unified"
)

fig_net.show()